# Récupération de la data sur l'API de Riot Games
### Workflow
1. Charger/collecter les PUUID joueurs par région
2. Récupérer 1 match classé par joueur
3. Sauvegarder `recap` + `timeline`
4. Reprendre automatiquement au prochain `match<count>`

## Bibliothèques utilisées

In [1]:
import requests
import time #nous avonsle droit de faire "20 requests every 1 seconds(s)" et "100 requests every 2 minutes(s)"
from tqdm import tqdm #afficher la progression de la récupération des données
import json
import os
import re
import signal
import sys

## Récupération API: 
remarque: la clé API se reset tous les 24h, il faut donc la mettre à jour régulièrement mais il n est pas necessaire de mettre data dans un environnement virtuel, on peut la stocker dans une variable globale

In [2]:
api_key = 'RGAPI-c3ac6bf0-8503-4b4b-909b-ad66d3044f7e'

### 1. Charger/collecter les PUUID joueurs par région

In [3]:
def recup_puuid_summoners(NJOUEURS): 
    """
    args:
        NJOUEURS (int): nombre total de joueurs à récupérer 
    return:
        dict: dictionnaire contenant les joueurs récupérés, organisé par région
    goal:
        Récupérer les puuid de NJOUEURS joueurs répartis équitablement entre les régions EUW, KR et NA, en utilisant les endpoints de l'API Riot
    """
    # On stocker par région, ça nous servira dans l'étape suivante pour récupérer les matchs de chaque joueur
    urls_summoners = {region: [] for platform,region in REGIONS.items()}
    for platform, region in REGIONS.items():
        for rang in RANGS:
            for division in DIVISIONS:
                url = f"https://{platform}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{rang}/{division}?page=1"
                urls_summoners[region].append(url)

    puuid_summoners = {region: [] for platform,region in REGIONS.items()}
    seen = set() # pour éviter les doublons de joueurs (un joueur peut être présent dans plusieurs pages s'il a changé de rang ou de division récemment)

    for region in tqdm(urls_summoners, desc="Régions"):
        for url in tqdm(urls_summoners[region], desc=f"URLs pour la région {region}"):
            if len(puuid_summoners[region]) >= NJOUEURS // len(REGIONS): # répartir équitablement les joueurs par région
                break # on s'arrête dès qu'on a assez (normalement on dépasse légèrement car 5000/3<1667)

            response = requests.get(url, headers={"X-Riot-Token": api_key})
            if response.status_code == 200: # succès : accès à la ressource
                data = response.json() #recupération des données au format json
                #print(data[0]["puuid"], url) # affichage du nom du premier joueur de la page et de l'url pour vérifier que ça fonctionne
                for i in range(NJOUEURS//60+1): # on peut faire 17 joueurs de pour les 60 différentes combinaisons pour atteindre les 1000 joueurs
                    s = data[i] if i < len(data) else None
                    sid = s.get("puuid")
                    if sid and sid not in seen:
                        seen.add(sid)
                        puuid_summoners[region].append(s)
                        if len(puuid_summoners[region]) >= NJOUEURS // len(REGIONS):
                            break
            else:
                print(f"Erreur {response.status_code} : {response.text}")

            time.sleep(1.2) # On peut faire 100 requètes toutes les 2min donc 1 requète toutes les 1.2s

    print(f"Nombre de joueurs récupérés : {sum(len(puuid_summoners[region]) for region in puuid_summoners)}")
    print({region: len(puuid_summoners[region]) for region in puuid_summoners})
    return puuid_summoners


def save_puuid_summoners(puuid_summoners: dict, fichier_index: int):
    """
    args:
        puuid_summoners (dict): dictionnaire contenant les joueurs à sauvegarder, organisé par région
        fichier_index (int): index du fichier à sauvegarder 
    return:
        None
    goal:
        Sauvegarde un bloc de joueurs dans data/puuid_summoners_{fichier_index}.json
    """
    path = os.path.join("data", f"puuid_summoners_{fichier_index}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(puuid_summoners, f, ensure_ascii=False, indent=2)
    total = sum(len(v) for v in puuid_summoners.values())
    print(f"{total} joueurs sauvegardés → {path}")


def load_puuid_summoners(count):
    """
    args:
        count (int): le count actuel de la collecte des matchs 
    return:
         dict: le contenu du fichier puuid_summoners correspondant au count actuel
    goal:
        Récupère le bon fichier puuid_summoners en fonction du count actuel
    """

    fichier_index = count // NJOUEURS
    path = os.path.join("data", f"puuid_summoners_{fichier_index}.json")

    if not os.path.exists(path):
        print(f"[INFO] Fichier {path} introuvable → collecte en cours...")
        puuid_summoners = recup_puuid_summoners(NJOUEURS)
        save_puuid_summoners(puuid_summoners, fichier_index)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def joueur_index_dans_fichier(count):
    """
    args:
        count (int): le count actuel de la collecte des matchs (ex: count=6 => 6ème match à collecter)
    return:
         int: l'index du joueur dans le fichier puuid_summoners correspondant au count actuel
    goal:
        recuopérer l'index du joueur dans le fichier puuid_summoners à partir de count
    """
    count_dans_fichier = count % NJOUEURS
    return count_dans_fichier // 3


### 2. Récupérer 1 match classé par joueur & 3. Sauvegarder `recap` + `timeline`

In [4]:
seen_matches = set()


def save_match(count, region, recap, timeline):
    """
    args:
        count (int): le count actuel de la collecte des matchs 
        region (str): la région du match (ex: "EU", "NA" ou "KR")
        recap (dict): le récapitulatif du match récupéré depuis l'API Riot
        timeline (dict): la timeline du match récupérée depuis l'API Riot
    return:
        None
    goal:
        Sauvegarde recap + timeline dans data/{region}/match{count}/
    """
    dir_path = os.path.join("data", region, f"match{count}")
    os.makedirs(dir_path, exist_ok=True)
    with open(os.path.join(dir_path, "recap.json"), "w") as f:
        json.dump(recap, f, indent=2)
    with open(os.path.join(dir_path, "timeline.json"), "w") as f:
        json.dump(timeline, f, indent=2)


def recupMatch(count, puuid_summoners,seen_matches):
    """
    args:
        count (int): le count actuel de la collecte des matchs
        puuid_summoners (dict): dictionnaire contenant les joueurs à récupérer, organisé par région
        seen_matches (set): ensemble des match_id déjà vus pour éviter les doublons
    return:
        bool: True si le match a été récupéré et sauvegardé avec succès, False si il n'y a plus de joueurs disponibles pour la région correspondante
    goal:
        Récupérer un match classé pour le joueur correspondant au count actuel, en utilisant les endpoints de l'API Riot, et le sauvegarder avec sa timeline. Si il n'y a plus de joueurs disponibles pour la région correspondante, retourner False pour arrêter la collecte.
    """
    # count % 3 → 0=europe, 1=americas, 2=asia
    regions = ["europe", "americas", "asia"]
    region  = regions[count % 3]

    # le joueur correspondant = count // 3  (ex: count=6 → joueur n°2 en europe)
    joueur_index = joueur_index_dans_fichier(count)
    puuids = puuid_summoners[region]

    if joueur_index >= len(puuids):
        print(f"[FIN] Plus de joueurs disponibles pour {region}")
        return False  

    puuid = puuids[joueur_index]["puuid"]

    #Récupère le match_id
    url = (
        f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/"
        f"{puuid}/ids?start=0&count=1&queue=420&type=ranked"
    )
    resp = requests.get(url, headers={"X-Riot-Token": api_key})
    time.sleep(1.2)

    if resp.status_code != 200:
        print(f"Erreur {resp.status_code} pour {puuid} : {resp.text}")
        return True

    ids = resp.json()
    if not ids:
        print(f"Aucun match trouvé pour {puuid}")
        return True

    match_id = ids[0]
    if match_id in seen_matches:
        print(f"Match {match_id} déjà vu, skip")
        return True
    seen_matches.add(match_id)

    #Récupère le recap
    url_recap = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    resp_recap = requests.get(url_recap, headers={"X-Riot-Token": api_key})
    time.sleep(1.2)

    #Récupère la timeline
    url_timeline = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}/timeline"
    resp_timeline = requests.get(url_timeline, headers={"X-Riot-Token": api_key})
    time.sleep(1.2)

    if resp_recap.status_code == 200 and resp_timeline.status_code == 200:
        save_match(count, region, resp_recap.json(), resp_timeline.json())
        print(f" match{count} ({region}) → {match_id}  [joueur #{joueur_index}]")
    else:
        print(f"Erreur recap={resp_recap.status_code} timeline={resp_timeline.status_code}")

    return True

### 4. Reprendre automatiquement au prochain `match<count>`

In [5]:
def load_counter():
    """
    args:
        None
    return:
        int: le count à partir duquel reprendre la collecte des matchs
    goal:
        Reprend le count à partir du plus grand match{N} trouvé dans data/
    """
    max_count = -1
    for region in ["europe", "americas", "asia"]:
        region_path = os.path.join("data", region)
        if not os.path.exists(region_path):
            continue
        for folder in os.listdir(region_path):
            match = re.match(r"^match(\d+)$", folder)
            if match:
                n = int(match.group(1))
                if n > max_count:
                    max_count = n

    if max_count == -1:
        return 0 # aucun match trouvé → on repart de 0
    return max_count + 1 # on reprend AU SUIVANT du dernier enregistré


### Main a éxécuter

In [6]:
# Nombre de Joueurs que je veux stocker par fichier puuid_summoners_0.json
NJOUEURS = 3000

REGIONS = {
    "euw1":  "europe",
    "na1":   "americas",
    "kr":    "asia",
}

RANGS = ["BRONZE", "SILVER", "GOLD", "PLATINUM", "DIAMOND"] #, "MASTER", "GRANDMASTER", "CHALLENGER"] #on ne s'intéresse pas à MASTER, GRANDMASTER et CHALLENGER car cela représente une très petite partie de la population de joueurs (moins de 1%) et que cela pourrait introduire un biais dans notre jeu de données. De plus, ces joueurs ont souvent des comportements de jeu très différents des joueurs de rangs inférieurs, ce qui pourrait également introduire un biais dans notre analyse.
DIVISIONS = ["I", "II", "III", "IV"]  


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":

    #créer les dossiersdata et REGION si n'existent pas encore
    os.makedirs("data", exist_ok=True)
    for region in REGIONS.values():
        os.makedirs(os.path.join("data", region), exist_ok=True)

    # reprendre counter au numero de matchs max
    count = load_counter()
    print(f"Reprise au match n°{count}")

    # Compteur de session pour le bilan final
    session_saves = 0

    # affiche le bilan proprement lorsqu'on arrete l'enregistrement des données'
    def on_exit(sig, frame):
        print(f"\nSession terminée — {session_saves} matchs sauvegardés (dernier : match{count - 1})")
        sys.exit(0)
    signal.signal(signal.SIGINT,  on_exit)
    signal.signal(signal.SIGTERM, on_exit)

    #Boucle principale
    seen_matches = set()
    while True:

        # Charge le bon fichier de joueurs (le collecte si besoin)
        puuid_summoners = load_puuid_summoners(count)  # gère auto la collecte + save

        #Récupère et sauvegarde le match
        ok = recupMatch(count, puuid_summoners, seen_matches)
        if not ok:
            print("Tous les joueurs ont été traités.")
            break

        count+= 1
        session_saves += 1


Reprise au match n°6000
 match6000 (europe) → EUW1_7709707613  [joueur #0]
 match6001 (americas) → NA1_5497280102  [joueur #0]
 match6002 (asia) → KR_8151194128  [joueur #0]
 match6003 (europe) → EUW1_7805430528  [joueur #1]
 match6004 (americas) → NA1_5466950404  [joueur #1]
 match6005 (asia) → KR_8106634282  [joueur #1]
 match6006 (europe) → EUW1_7807718733  [joueur #2]
 match6007 (americas) → NA1_5524811588  [joueur #2]
 match6008 (asia) → KR_8158807666  [joueur #2]
 match6009 (europe) → EUW1_7801606045  [joueur #3]
 match6010 (americas) → NA1_5488274104  [joueur #3]
 match6011 (asia) → KR_8128845762  [joueur #3]
 match6012 (europe) → EUW1_7737416338  [joueur #4]
 match6013 (americas) → NA1_5528445681  [joueur #4]
 match6014 (asia) → KR_8155318921  [joueur #4]
 match6015 (europe) → EUW1_7799535355  [joueur #5]
 match6016 (americas) → NA1_5476931726  [joueur #5]
 match6017 (asia) → KR_8159160563  [joueur #5]
 match6018 (europe) → EUW1_7791653073  [joueur #6]
 match6019 (americas) → N

SystemExit: 0

C:\Users\Alexis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
